<a href="https://colab.research.google.com/github/hongwoomin02/capstone-deepfake/blob/h8week/8%EC%A3%BC%EC%B0%A8_%EC%98%81%EC%83%81%ED%95%99%EC%8A%B5_4_21v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# [셀 2~3 대체] kagglehub로 데이터 다운로드
# ============================================================================

import kagglehub
import os

print("📥 데이터 다운로드 시작... (10~20분 소요)")
path = kagglehub.dataset_download("muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal")

print(f"\n✅ 다운로드 완료")
print(f"경로: {path}")

# DATA_DIR 변수 설정 (뒤 셀에서 사용)
DATA_DIR = path

# 폴더 구조 확인
print(f"\n=== 데이터 구조 ===")
for root, dirs, files in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f'{indent}📁 {os.path.basename(root) if depth > 0 else "ROOT"}/')
    for f in files[:3]:
        print(f'{indent}   📄 {f}')
    if len(files) > 3:
        print(f'{indent}   ... ({len(files)} files)')

In [ ]:
"""
=============================================================================
  GRAVEX-200K 딥페이크 탐지 학습 — Google Colab 버전

  환경: Colab Pro (A100/V100/T4) 권장, 무료 T4도 가능

  실행 순서:
    셀 1: 환경 세팅 + Drive 마운트
    셀 2: Kaggle API 설정
    셀 3: 데이터셋 다운로드
    셀 4: 임포트 + 경로 설정
    셀 5: CSV 로드 + 소스 분류
    셀 6: Dataset 클래스
    셀 7: DataLoader
    셀 8: 모델 정의
    셀 9: Trainer 클래스
    셀 10: 빠른 검증 학습 (10%, 3ep)
    셀 11: 본 학습 (100%, 20ep)
    셀 12: 체크포인트 이어 학습 (필요 시)
    셀 13: 학습 곡선
    셀 14: Test 평가
    셀 15: 소스별 분석
    셀 16: 결과 저장
=============================================================================
"""

# ============================================================================
# [셀 1] Google Drive 마운트 + 작업 폴더 생성
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

import os

# Drive에 프로젝트 폴더 생성 (체크포인트 저장용)
PROJECT_DIR = '/content/drive/MyDrive/capstone_deepfake'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)

print(f"✅ Drive 마운트 완료")
print(f"✅ 프로젝트 폴더: {PROJECT_DIR}")

# GPU 확인
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # 현재 사용 중인 메모리
    print(f"할당된 VRAM: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")


In [ ]:
pip install -q timm

In [ ]:
# [셀 4] 임포트 + 데이터 경로 설정
# ============================================================================

import os
import io
import re
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter
from PIL import Image


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# 시드 고정
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ⚠️ 실제 다운로드된 폴더 구조를 확인하고 경로 맞추기
# 보통 다음 중 하나:
#   /content/data/my_real_vs_ai_dataset/my_real_vs_ai_dataset/
#   /content/data/

# 자동 탐색
print("=== 데이터 구조 탐색 ===")
for root, dirs, files in os.walk(DATA_DIR):
    # 너무 깊게 가지 않도록 제한
    depth = root.replace(DATA_DIR, '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f'{indent}📁 {os.path.basename(root)}/')
    for f in files[:3]:
        print(f'{indent}   📄 {f}')
    if len(files) > 3:
        print(f'{indent}   ... ({len(files)} files total)')


In [ ]:
# ============================================================================
# [셀 5] 경로 확정 + CSV 로드 + 소스 분류
# ============================================================================

# ⚠️ 셀 4 출력을 보고 아래 경로 수정
# 보통 압축 해제하면 my_real_vs_ai_dataset 중첩 구조

NESTED = os.path.join(DATA_DIR, "my_real_vs_ai_dataset", "my_real_vs_ai_dataset")

# CSV는 최상위
TRAIN_CSV = os.path.join(DATA_DIR, "train_labels.csv")
VAL_CSV   = os.path.join(DATA_DIR, "val_labels.csv")
TEST_CSV  = os.path.join(DATA_DIR, "test_labels.csv")

# 이미지는 label별 폴더
AI_IMG_DIR   = os.path.join(NESTED, "ai_images")    # Label 0 (Fake)
REAL_IMG_DIR = os.path.join(NESTED, "real")          # Label 1 (Real)

# 경로 존재 확인
for name, path in [
    ("TRAIN_CSV", TRAIN_CSV), ("VAL_CSV", VAL_CSV), ("TEST_CSV", TEST_CSV),
    ("AI_IMG_DIR", AI_IMG_DIR), ("REAL_IMG_DIR", REAL_IMG_DIR)
]:
    status = "✅" if os.path.exists(path) else "❌"
    print(f"{status} {name}: {path}")

# CSV 로드
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"\nTrain: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}")

# 소스 분류 함수 (정규식 미리 컴파일 → 빠름)
_pat_alphanum = re.compile(r'^[A-Z0-9]{10}\.jpg$', re.IGNORECASE)
_pat_numeric  = re.compile(r'^\d+\.jpg$')
_pat_sd       = re.compile(r'^(512|768|1024)_')

def classify_source(filename):
    fname = filename.lower()
    if 'dfdc' in fname and 'fake' in fname:
        return 'DFDC_fake'
    elif 'dfdc' in fname:
        return 'DFDC_real'
    elif 'celebdf' in fname and ('fake' in fname or 'id' in fname):
        return 'CelebDF_fake'
    elif 'celebdf' in fname:
        return 'CelebDF_real'
    elif fname.startswith('fake_deepfakes'):
        return 'FF++_Deepfakes'
    elif fname.startswith('fake_faceswap'):
        return 'FF++_FaceSwap'
    elif fname.startswith('fake_face2face'):
        return 'FF++_Face2Face'
    elif fname.startswith('fake_neuraltextures'):
        return 'FF++_NeuralTextures'
    elif fname.startswith('fake_faceshifter'):
        return 'FF++_FaceShifter'
    elif fname.startswith('fake_deepfacelab'):
        return 'FF++_DeepFaceLab'
    elif fname.startswith('real_'):
        return 'FF++_Real'
    elif _pat_sd.match(fname):
        return 'StableDiffusion'
    elif fname.startswith('aug_'):
        return 'Augmented_AI'
    elif fname.startswith('ai_detect'):
        return 'AI_Detection_HQ'
    elif _pat_alphanum.match(filename):
        return '140k_RealvsFake'
    elif _pat_numeric.match(fname):
        return '140k_RealvsFake'
    else:
        return 'Other'

train_df['source'] = train_df['filename'].apply(classify_source)
val_df['source']   = val_df['filename'].apply(classify_source)
test_df['source']  = test_df['filename'].apply(classify_source)

print("\n=== Train 소스별 분포 ===")
print(train_df.groupby(['source', 'label']).size().unstack(fill_value=0))

In [ ]:
# ============================================================================
# [셀 6] Dataset 클래스 + Transform
# ============================================================================

class RandomJPEGCompression:
    """JPEG 압축 노이즈 랜덤 추가"""
    def __init__(self, quality_range=(30, 90)):
        self.quality_range = quality_range

    def __call__(self, img):
        quality = random.randint(*self.quality_range)
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert('RGB')


class GRAVEXDataset(Dataset):
    """GRAVEX-200K — ai_images/ + real/ 두 폴더 구조"""

    def __init__(self, csv_path, ai_img_dir, real_img_dir, transform=None,
                 verify_paths=False):
        self.df = pd.read_csv(csv_path)
        self.ai_img_dir   = ai_img_dir
        self.real_img_dir = real_img_dir
        self.transform = transform
        self.df['source'] = self.df['filename'].apply(classify_source)

        # verify_paths=True면 유효성 검증 (시간 오래 걸림, 기본 생략)
        if verify_paths:
            def get_path(row):
                folder = ai_img_dir if row['label'] == 0 else real_img_dir
                return os.path.join(folder, row['filename'])
            valid = self.df.apply(lambda row: os.path.exists(get_path(row)), axis=1)
            missing = (~valid).sum()
            if missing > 0:
                print(f"⚠️ {missing}개 이미지 없음 — 제외됨")
            self.df = self.df[valid].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        folder = self.ai_img_dir if row['label'] == 0 else self.real_img_dir
        img_path = os.path.join(folder, row['filename'])

        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            # 손상된 이미지는 검은 이미지로 대체 (학습 중단 방지)
            print(f"⚠️ 이미지 로드 실패: {img_path} ({e})")
            img = Image.new('RGB', (256, 256), (0, 0, 0))

        if self.transform:
            img = self.transform(img)

        return img, int(row['label'])

    def get_source(self, idx):
        return self.df.iloc[idx]['source']


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    RandomJPEGCompression(quality_range=(30, 90)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


In [ ]:
# ============================================================================
# [셀 7] DataLoader 생성
# ============================================================================

print("데이터셋 생성 중...")
train_dataset = GRAVEXDataset(TRAIN_CSV, AI_IMG_DIR, REAL_IMG_DIR, transform=train_transform)
val_dataset   = GRAVEXDataset(VAL_CSV,   AI_IMG_DIR, REAL_IMG_DIR, transform=eval_transform)
test_dataset  = GRAVEXDataset(TEST_CSV,  AI_IMG_DIR, REAL_IMG_DIR, transform=eval_transform)

print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

# Colab GPU별 권장 설정
# T4 (16GB): BATCH=32, NUM_WORKERS=2
# V100/A100: BATCH=64, NUM_WORKERS=4
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
if 'A100' in gpu_name or 'V100' in gpu_name:
    BATCH_SIZE = 48
    NUM_WORKERS = 4
else:
    BATCH_SIZE = 24
    NUM_WORKERS = 2

print(f"\nBatch size: {BATCH_SIZE} | Workers: {NUM_WORKERS}")

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=True  # Colab에서 더 안정적
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

# 배치 테스트
print("\n배치 로드 테스트...")
images, labels = next(iter(train_loader))
print(f"✅ Images: {images.shape}, Labels: {labels.shape}")


In [ ]:
# ============================================================================
# [셀 8] 모델 정의
# ============================================================================

# 셀 8 build_model 함수 전체를 이걸로 교체

def build_model(num_classes=2, pretrained=True):
    """ConvNeXt V2-Tiny 기반 딥페이크 탐지 모델"""

    # 방법 1: timm 라이브러리 사용 (권장, FCMAE 사전학습 가중치)
    import timm

    model = timm.create_model(
        'convnextv2_tiny.fcmae_ft_in22k_in1k',  # ImageNet-22k로 사전학습 후 1k로 fine-tune
        pretrained=pretrained,
        num_classes=num_classes,
        drop_rate=0.1,       # Dropout
        drop_path_rate=0.1,  # Stochastic Depth
    )

    return model




model = build_model(num_classes=2, pretrained=True).to(device)
print(f"Model: convnextv2-tiny")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# ============================================================================
# [셀 9] Trainer 클래스
# ============================================================================

class DeepfakeTrainer:
    def __init__(self, model, device, save_dir):
        self.model = model
        self.device = device
        self.save_dir = save_dir
        self.criterion = nn.CrossEntropyLoss()
        # ConvNeXt에 맞게 수정
        self.optimizer = optim.AdamW(
              model.parameters(),
              lr=5e-5,            # 1e-4 → 5e-5로 낮춤
              weight_decay=0.05   # ConvNeXt 논문 권장값
          )
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='max', factor=0.5, patience=2
        )
        self.history = {
            'train_loss': [], 'train_acc': [], 'train_f1': [],
            'val_loss': [], 'val_acc': [], 'val_f1': [],
            'val_recall': [], 'val_precision': [], 'val_auc': [],
            'lr': [], 'epoch_time': []
        }
        self.best_f1 = 0.0
        self.patience_counter = 0

    def train_one_epoch(self, loader):
        self.model.train()
        running_loss = 0.0
        all_preds, all_labels = [], []

        for batch_idx, (images, labels) in enumerate(loader):
            images, labels = images.to(self.device), labels.to(self.device)
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            if (batch_idx + 1) % 200 == 0:
                print(f"  Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}")

        avg_loss = running_loss / len(loader)
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='binary', pos_label=0)
        return avg_loss, acc, f1

    @torch.no_grad()
    def evaluate(self, loader):
        self.model.eval()
        running_loss = 0.0
        all_preds, all_labels, all_probs = [], [], []

        for images, labels in loader:
            images, labels = images.to(self.device), labels.to(self.device)
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            running_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

        all_probs_np = np.array(all_probs)
        avg_loss = running_loss / len(loader)
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='binary', pos_label=0)
        rec = recall_score(all_labels, all_preds, average='binary', pos_label=0)
        prec = precision_score(all_labels, all_preds, average='binary', pos_label=0)

        try:
            auc = roc_auc_score([1 - l for l in all_labels], all_probs_np[:, 0])
        except:
            auc = 0.0

        return {
            'loss': avg_loss, 'accuracy': acc, 'f1': f1,
            'recall': rec, 'precision': prec, 'auc': auc,
            'preds': np.array(all_preds), 'labels': np.array(all_labels),
            'probs': all_probs_np
        }

    def train(self, train_loader, val_loader, epochs=20, patience=3):
        print("=" * 60)
        print(f"학습 시작 | Epochs: {epochs} | Patience: {patience}")
        print(f"저장 경로: {self.save_dir}")
        print("=" * 60)

        for epoch in range(1, epochs + 1):
            t0 = time.time()
            print(f"\n--- Epoch {epoch}/{epochs} ---")

            train_loss, train_acc, train_f1 = self.train_one_epoch(train_loader)
            val_results = self.evaluate(val_loader)

            self.scheduler.step(val_results['f1'])
            lr = self.optimizer.param_groups[0]['lr']
            elapsed = time.time() - t0

            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['train_f1'].append(train_f1)
            self.history['val_loss'].append(val_results['loss'])
            self.history['val_acc'].append(val_results['accuracy'])
            self.history['val_f1'].append(val_results['f1'])
            self.history['val_recall'].append(val_results['recall'])
            self.history['val_precision'].append(val_results['precision'])
            self.history['val_auc'].append(val_results['auc'])
            self.history['lr'].append(lr)
            self.history['epoch_time'].append(elapsed)

            print(f"  Train | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
            print(f"  Val   | Acc: {val_results['accuracy']:.4f} | F1: {val_results['f1']:.4f} | "
                  f"Recall: {val_results['recall']:.4f} | AUC: {val_results['auc']:.4f}")
            print(f"  Time: {elapsed:.0f}s ({elapsed/60:.1f}min) | LR: {lr:.6f}")

            # Best model (Drive에 저장 → 세션 끊겨도 유지)
            if val_results['f1'] > self.best_f1:
                self.best_f1 = val_results['f1']
                self.patience_counter = 0
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'best_f1': self.best_f1,
                    'history': self.history,
                }, os.path.join(self.save_dir, "best_model_gravex.pt"))
                print(f"  ✅ Best saved! (F1: {self.best_f1:.4f})")
            else:
                self.patience_counter += 1
                print(f"  ⏳ No improvement ({self.patience_counter}/{patience})")

            # 체크포인트 (매 epoch마다 Drive에 저장 → 세션 끊김 대비)
            torch.save({
                'epoch': epoch,
                'model_state_dict': self.model.state_dict(),
                'optimizer_state_dict': self.optimizer.state_dict(),
                'best_f1': self.best_f1,
                'history': self.history,
            }, os.path.join(self.save_dir, f"ckpt_latest.pt"))

            # 3 epoch마다 별도 체크포인트
            if epoch % 3 == 0:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'best_f1': self.best_f1,
                    'history': self.history,
                }, os.path.join(self.save_dir, f"ckpt_ep{epoch}.pt"))

            if self.patience_counter >= patience:
                print(f"\n🛑 Early stopping")
                break

        # History 저장
        h = {k: [float(v) for v in vals] for k, vals in self.history.items()}
        with open(os.path.join(self.save_dir, "training_history.json"), 'w') as f:
            json.dump(h, f, indent=2)

        return self.history

In [ ]:
# #============================================================================
# # [셀 10] 빠른 검증 학습 (필수!)
# # ============================================================================

# subset_size = len(train_dataset) // 10
# val_sub_size = len(val_dataset) // 10
# subset_train = Subset(train_dataset, random.sample(range(len(train_dataset)), subset_size))
# subset_val   = Subset(val_dataset,   random.sample(range(len(val_dataset)), val_sub_size))

# sub_train_loader = DataLoader(
#     subset_train, batch_size=BATCH_SIZE, shuffle=True,
#     num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=True
# )
# sub_val_loader = DataLoader(
#     subset_val, batch_size=BATCH_SIZE, shuffle=False,
#     num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
# )

# print(f"빠른 검증: {subset_size:,} train / {val_sub_size:,} val")

# quick_model = build_model(num_classes=2, pretrained=True).to(device)
# quick_save = f'{PROJECT_DIR}/checkpoints/quick_test'
# os.makedirs(quick_save, exist_ok=True)
# quick_trainer = DeepfakeTrainer(quick_model, device, quick_save)

# quick_history = quick_trainer.train(sub_train_loader, sub_val_loader, epochs=3, patience=10)

# # 시간 예측
# avg_time = np.mean(quick_history['epoch_time'])
# full_est = avg_time * 10
# print(f"\n{'='*50}")
# print(f"10% 기준 1 epoch: {avg_time:.0f}초 ({avg_time/60:.1f}분)")
# print(f"전체 1 epoch 예상: {full_est:.0f}초 ({full_est/60:.1f}분)")
# print(f"20 epoch 총 예상: {full_est*20/3600:.1f}시간")

# # Colab 세션 한도
# # 무료: 약 12시간 (실사용 4~6시간 후 끊기는 경우도 있음)
# # Pro: 24시간
# if full_est * 20 / 3600 > 10:
#     print(f"⚠️ 10시간 초과 — 중간 체크포인트에서 이어 학습 필요할 수 있음")
# else:
#     print(f"✅ Colab 세션 안에 완료 가능")

# del quick_model, quick_trainer
# torch.cuda.empty_cache()

In [ ]:
# ============================================================================
# [셀 11] 본 학습 (처음 실행 시)
# ============================================================================

SAVE_DIR = f'{PROJECT_DIR}/checkpoints/main'
os.makedirs(SAVE_DIR, exist_ok=True)

model = build_model(num_classes=2, pretrained=True).to(device)
trainer = DeepfakeTrainer(model, device, SAVE_DIR)

print("🚀 본 학습 시작")
history = trainer.train(train_loader, val_loader, epochs=20, patience=3)

In [ ]:
# ============================================================================
# [셀 12] 체크포인트 이어 학습 (세션 끊겨서 재시작할 때)
# ============================================================================

"""
세션이 끊기거나 런타임이 죽으면:
  1. 새 Colab 세션 시작
  2. 셀 1~9 다시 실행 (빠름, 데이터는 /content에 없음)
  3. 데이터 다시 다운로드 필요 → 셀 3 재실행
  4. 이 셀로 이어 학습
"""

# === 주석 해제해서 사용 ===

# CKPT_PATH = f'{PROJECT_DIR}/checkpoints/main/ckpt_latest.pt'
#
# if os.path.exists(CKPT_PATH):
#     model = build_model(num_classes=2, pretrained=False).to(device)
#     ckpt = torch.load(CKPT_PATH, map_location=device)
#     model.load_state_dict(ckpt['model_state_dict'])
#
#     trainer = DeepfakeTrainer(model, device, SAVE_DIR)
#     trainer.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
#     trainer.best_f1 = ckpt['best_f1']
#     trainer.history = ckpt['history']
#
#     start_epoch = ckpt['epoch']
#     remaining = 20 - start_epoch
#     print(f"체크포인트 로드: epoch {start_epoch}, best F1: {trainer.best_f1:.4f}")
#     print(f"남은 epochs: {remaining}")
#
#     history = trainer.train(train_loader, val_loader, epochs=remaining, patience=3)
# else:
#     print(f"❌ 체크포인트 없음: {CKPT_PATH}")


In [ ]:
# ============================================================================
# [셀 13] 학습 곡선 시각화
# ============================================================================

def plot_curves(history, save_path):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("GRAVEX-200K Training Curves", fontsize=16, fontweight='bold')
    ep = range(1, len(history['train_loss']) + 1)

    axes[0,0].plot(ep, history['train_loss'], 'b-', label='Train', lw=2)
    axes[0,0].plot(ep, history['val_loss'], 'r-', label='Val', lw=2)
    axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

    axes[0,1].plot(ep, history['train_acc'], 'b-', label='Train', lw=2)
    axes[0,1].plot(ep, history['val_acc'], 'r-', label='Val', lw=2)
    axes[0,1].set_title('Accuracy'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

    axes[0,2].plot(ep, history['train_f1'], 'b-', label='Train', lw=2)
    axes[0,2].plot(ep, history['val_f1'], 'r-', label='Val', lw=2)
    axes[0,2].set_title('F1 Score'); axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

    axes[1,0].plot(ep, history['val_recall'], 'g-', lw=2)
    axes[1,0].axhline(y=0.85, color='r', ls='--', alpha=0.5, label='목표 85%')
    axes[1,0].set_title('Val Recall (Fake 탐지율)'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

    axes[1,1].plot(ep, history['val_auc'], 'm-', lw=2)
    axes[1,1].set_title('Val ROC-AUC'); axes[1,1].grid(alpha=0.3)

    axes[1,2].plot(ep, history['lr'], 'k-', lw=2)
    axes[1,2].set_title('Learning Rate'); axes[1,2].set_yscale('log'); axes[1,2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=72, bbox_inches='tight')
    plt.show()

plot_curves(history, f'{PROJECT_DIR}/results/training_curves.png')

In [ ]:
# ============================================================================
# [셀 14] Test Set 평가
# ============================================================================

best = build_model(num_classes=2, pretrained=False).to(device)
ckpt = torch.load(f'{SAVE_DIR}/best_model_gravex.pt', map_location=device)
best.load_state_dict(ckpt['model_state_dict'])
print(f"Best model: epoch {ckpt['epoch']}, val F1: {ckpt['best_f1']:.4f}")

evaluator = DeepfakeTrainer(best, device, SAVE_DIR)
test_results = evaluator.evaluate(test_loader)

print("\n" + "=" * 60)
print("GRAVEX-200K Test 결과 (20,000장)")
print("=" * 60)
print(f"Accuracy:  {test_results['accuracy']:.4f}")
print(f"Precision: {test_results['precision']:.4f}")
print(f"Recall:    {test_results['recall']:.4f}")
print(f"F1:        {test_results['f1']:.4f}")
print(f"AUC:       {test_results['auc']:.4f}")

cm = confusion_matrix(test_results['labels'], test_results['preds'])
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)
labels_disp = ['Fake/AI (0)', 'Real (1)']
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(labels_disp); ax.set_yticklabels(labels_disp)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i,j] > cm.max()/2 else 'black'
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                fontsize=16, fontweight='bold', color=color)
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/confusion_matrix.png', dpi=72, bbox_inches='tight')
plt.show()

print("\n" + classification_report(
    test_results['labels'], test_results['preds'],
    target_names=['Fake/AI', 'Real']
))

In [ ]:
# ============================================================================
# [셀 15] 소스별 성능 분석
# ============================================================================

@torch.no_grad()
def evaluate_by_source(model, dataset, batch_size=32):
    model.eval()

    # 배치 단위로 빠르게
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

    all_preds, all_labels, all_sources = [], [], []
    idx = 0
    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        labels_np = labels.numpy()

        # 현재 배치의 소스 정보
        for i in range(len(labels_np)):
            all_sources.append(dataset.get_source(idx + i))

        all_preds.extend(preds)
        all_labels.extend(labels_np)
        idx += len(labels_np)

        if idx % 2000 == 0:
            print(f"  {idx}/{len(dataset)} 처리...")

    # 소스별 집계
    results_df = pd.DataFrame({
        'source': all_sources,
        'label': all_labels,
        'pred': all_preds
    })

    summary = {}
    for source, group in results_df.groupby('source'):
        labels = group['label'].values
        preds = group['pred'].values

        acc = accuracy_score(labels, preds)
        if len(np.unique(labels)) >= 2:
            f1 = f1_score(labels, preds, average='binary', pos_label=0, zero_division=0)
            rec = recall_score(labels, preds, average='binary', pos_label=0, zero_division=0)
            summary[source] = {
                'count': len(labels), 'accuracy': float(acc),
                'f1': float(f1), 'recall': float(rec)
            }
        else:
            summary[source] = {
                'count': len(labels), 'accuracy': float(acc),
                'dominant_label': int(labels[0])
            }

    return summary

print("소스별 평가 시작...")
source_results = evaluate_by_source(best, test_dataset, batch_size=BATCH_SIZE)

print("\n" + "=" * 70)
print("소스별 성능 분석")
print("=" * 70)
print(f"{'Source':<25} {'Count':>7} {'Accuracy':>10} {'F1':>8} {'Recall':>8}")
print("-" * 70)
for source in sorted(source_results.keys()):
    r = source_results[source]
    f1_s = f"{r.get('f1', 0):.4f}" if 'f1' in r else "N/A"
    rec_s = f"{r.get('recall', 0):.4f}" if 'recall' in r else "N/A"
    print(f"{source:<25} {r['count']:>7,} {r['accuracy']:>10.4f} {f1_s:>8} {rec_s:>8}")

# 시각화
sources = sorted(source_results.keys())
accs = [source_results[s]['accuracy'] for s in sources]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(len(sources)), accs, color='steelblue', alpha=0.8)
ax.set_yticks(range(len(sources)))
ax.set_yticklabels(sources, fontsize=10)
ax.set_xlabel('Accuracy')
ax.set_title('Performance by Data Source')
ax.axvline(x=0.85, color='r', ls='--', alpha=0.5, label='목표 85%')
ax.legend()
for i, v in enumerate(accs):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/source_analysis.png', dpi=72, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# [셀 16] 최종 결과 저장 + 팀원 B에게 전달할 파일 목록
# ============================================================================

results_json = {
    'model': 'ConvNextV2-tiny',
    'dataset': 'GRAVEX-200K',
    'label_mapping': {'0': 'Fake/AI', '1': 'Real'},
    'positive_class': '0 (Fake/AI)',
    'best_epoch': int(ckpt['epoch']),
    'internal_test': {
        'accuracy': float(test_results['accuracy']),
        'precision': float(test_results['precision']),
        'recall': float(test_results['recall']),
        'f1': float(test_results['f1']),
        'auc': float(test_results['auc']),
    },
    'source_analysis': source_results,
    'confusion_matrix': cm.tolist(),
    'timestamp': datetime.now().isoformat(),
}

with open(f'{PROJECT_DIR}/results/internal_eval_results.json', 'w') as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 60)
print("✅ 최종 산출물 (Google Drive에 저장됨)")
print("=" * 60)
for root, _, files in os.walk(PROJECT_DIR):
    for f in sorted(files):
        fp = os.path.join(root, f)
        size_mb = os.path.getsize(fp) / 1e6
        rel = fp.replace(PROJECT_DIR, '')
        print(f"  {rel} ({size_mb:.1f} MB)")

print(f"\n팀원 B에게 전달:")
print(f"  - {PROJECT_DIR}/checkpoints/main/best_model_gravex.pt")
print(f"  - {PROJECT_DIR}/results/internal_eval_results.json")
print(f"\n💡 Drive 공유 링크로 팀원 B와 공유하면 편해요!")